# Babel Gauge — Legacy Telemetry Log Canonicalization: reference solution

**Approach — format archaeology.** The six legacy grammars are undocumented,
so everything is inferred from the 60,000 training pairs:

1. **Sniff** structural signatures to cluster lines into format families.
2. **Learn** every hidden constant *from the pairs*: per-site UTC offsets,
   status vocabularies, device-type letter codes, numeric transforms
   (°F→°C, psi→kPa, implied ×10 decimals), and each format's date order —
   with printed evidence for every learned rule.
3. **Validate** the assembled transducer on held-out training pairs with the
   task metric before touching the test set.
4. **REJECT** = any line that matches no learned grammar (corrupted ingest).

No neural model is needed once the transduction rules are recovered exactly;
the hard part is *recovering them all* — each missed quirk costs measurable
score (shown in the baseline comparison below).

In [1]:
import re
import time
import numpy as np
import pandas as pd

T0 = time.time()
SEED = 42
DATA = "./dataset/public"

train = pd.read_csv(f"{DATA}/train.csv")
test = pd.read_csv(f"{DATA}/test.csv")
sites = pd.read_csv(f"{DATA}/sites.csv")
SITE_SET = set(sites.site)
SC = "|".join(sorted(SITE_SET, key=len, reverse=True))
print(train.shape, test.shape, len(SITE_SET), "sites")
train.head(3)

(60000, 3) (8000, 2) 36 sites


,line_id,raw_line,canonical
0,L000001,<C4664|ATH2|20250412035659|t:59.8|p:305.8|s:1>,ts=2025-04-12T03:56:59Z|dev=CMP-4664|site=ATH2...
1,L000002,time=22/03/2025 17:14 site=IND7 flag=WARN temp...,ts=2025-03-22T11:44:00Z|dev=CMP-1279|site=IND7...
2,L000003,V3965IND2 2502230054+0777001805O,ts=2025-02-22T19:24:00Z|dev=VLV-3965|site=IND2...


## 1 — Sniff the format families

Inspecting samples shows six recurring line shapes. Each gets a strict
full-line signature; anything matching none of them is a corruption
candidate. The signatures are deliberately strict (anchored, exact widths)
so that damaged lines fail cleanly instead of half-parsing.

In [2]:
SIGS = {
    "F1": re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) \| ([A-Z]{3})-(\d{4}) @ (%s) \| T=(-?\d+\.\d+)F P=(\d+\.\d+)psi \| ([A-Z]+)$" % SC),
    "F2": re.compile(r"^<([A-Z])(\d{4})\|(%s)\|(\d{14})\|t:(-?\d+\.\d)\|p:(\d+\.\d)\|s:(\d)>$" % SC),
    "F4": re.compile(r"^([A-Z])(\d{4})(%s) *(\d{6})(\d{4})([+-]\d{4})(\d{6})([A-Z])$" % SC),
    "F5": re.compile(r"^(%s),([A-Z]{3})-(\d{4}),(\d{2}/\d{2}/\d{2} \d{2}:\d{2}[AP]M),(-?\d+\.\d+),(\d+\.\d+),([A-Z]{3})$" % SC),
    "F6": re.compile(r'^\{"st":"(%s)","d":"([A-Z]{3})-(\d{4})","e":(\d+),"tc":(-?\d+\.\d),"pk":(-?\d+\.\d),"x":"([a-z]+)"\}$' % SC),
}


def f3_kv(raw):
    toks = raw.split(" ")
    kv, i = {}, 0
    while i < len(toks):
        if "=" not in toks[i]:
            return None
        k, v = toks[i].split("=", 1)
        if k == "time":
            if i + 1 >= len(toks):
                return None
            v = v + " " + toks[i + 1]
            i += 1
        kv[k] = v
        i += 1
    return kv


def sniff(raw):
    for name, rx in SIGS.items():
        if rx.match(raw):
            return name
    kv = f3_kv(raw)
    if kv is not None and set(kv) == {"dev", "site", "time", "temp",
                                      "pressure", "flag"}:
        return "F3"
    return None


train["fmt"] = train.raw_line.map(sniff)
lab = train[train.canonical != "REJECT"]
rej = train[train.canonical == "REJECT"]
print("format counts (clean lines):", lab.fmt.value_counts(dropna=False).to_dict())
print("clean lines matching no grammar:", lab.fmt.isna().sum())
print("REJECT lines matching a grammar:", rej.fmt.notna().sum())
for f in ["F1", "F2", "F3", "F4", "F5", "F6"]:
    print(f, "|", lab[lab.fmt == f].raw_line.iloc[0])

format counts (clean lines): {'F1': 16182, 'F2': 12822, 'F3': 11614, 'F4': 10156, 'F5': 6011, 'F6': 855}
clean lines matching no grammar: 0
REJECT lines matching a grammar: 0
F1 | 2025-04-30 16:06:09 | VLV-2004 @ UAE3 | T=170.1F P=45.22psi | OK
F2 | <C4664|ATH2|20250412035659|t:59.8|p:305.8|s:1>
F3 | time=22/03/2025 17:14 site=IND7 flag=WARN temp=44.1C pressure=156.2kPa dev=CMP1279
F4 | V3965IND2 2502230054+0777001805O
F5 | SIN2,VLV-4675,02/08/25 07:07PM,172.6,94.48,WRN
F6 | {"st":"IRL2","d":"CMP-5862","e":1747296976,"tc":46.6,"pk":325.0,"x":"ok"}


## 2 — Learn the hidden constants from the pairs

Parse the canonical side of each training pair and align it with the raw
side, format by format:

- **status vocabularies** — contingency of raw status token vs canonical
  status (must be unambiguous);
- **device-type letters** (F2/F4 encode `PMP/CMP/FAN/VLV` as one char);
- **per-site UTC offsets** — mode of (raw local time − canonical UTC time)
  per site, learned from F1's full datetimes, then cross-checked against
  the provided `sites.csv`;
- **numeric transforms** — for each numeric field, pick among candidate
  transforms (identity, ×0.1, °F→°C, psi→kPa) the one that reproduces the
  canonical value exactly after rounding;
- **date order** — for F3 and F5, try both DD/MM and MM/DD against the
  canonical timestamp (offset-corrected) and count exact hits.

In [3]:
def canon_kv(c):
    return dict(p.split("=", 1) for p in c.split("|"))


ckv = lab.canonical.map(canon_kv)
lab = lab.assign(
    c_dev=[k["dev"] for k in ckv], c_site=[k["site"] for k in ckv],
    c_tc=[k["temp_c"] for k in ckv], c_pk=[k["pres_kpa"] for k in ckv],
    c_st=[k["status"] for k in ckv],
    c_ts_dt=pd.to_datetime([k["ts"][:-1] for k in ckv]))

LEARN, TYPEMAP = {}, {}


def learn_status(sub, getter, fmt):
    m = {}
    for rs, cs in zip(sub.raw_line.map(getter), sub.c_st):
        m.setdefault(rs, {}).setdefault(cs, 0)
        m[rs][cs] += 1
    assert all(len(v) == 1 for v in m.values()), f"ambiguous status map {fmt}: {m}"
    table = {k: max(v, key=v.get) for k, v in m.items()}
    print(f"  {fmt} status map: {table}")
    return table


def learn_offset(sub, local_getter):
    d = {}
    for raw, cts, site in zip(sub.raw_line, sub.c_ts_dt, sub.c_site):
        mins = int((local_getter(raw) - cts).total_seconds() // 60)
        d.setdefault(site, {}).setdefault(mins, 0)
        d[site][mins] += 1
    return {s: max(v, key=v.get) for s, v in d.items()}


NUM_T = {"identity": lambda x: x, "x/10": lambda x: x / 10,
         "F->C": lambda x: (x - 32) * 5 / 9,
         "psi->kPa": lambda x: x * 6.894757}


def learn_num(sub, getter, target, fmt, field):
    x = np.array([float(getter(r)) for r in sub.raw_line])
    y = target.astype(float).to_numpy()
    errs = {n: np.abs(np.round(f(x), 1) - y).max() for n, f in NUM_T.items()}
    best = min(errs, key=errs.get)
    print(f"  {fmt}.{field}: {best} (max err {errs[best]:.4f})")
    assert errs[best] < 0.051, f"no exact transform for {fmt}.{field}"
    return best


# F1 — full local datetime, imperial units
sub = lab[lab.fmt == "F1"]
g1 = SIGS["F1"]
LEARN["F1_status"] = learn_status(sub, lambda r: g1.match(r).group(7), "F1")
LEARN["F1_tc"] = learn_num(sub, lambda r: g1.match(r).group(5), sub.c_tc, "F1", "temp")
LEARN["F1_pk"] = learn_num(sub, lambda r: g1.match(r).group(6), sub.c_pk, "F1", "pres")
OFFSETS = learn_offset(sub, lambda r: pd.Timestamp(g1.match(r).group(1)))
prov = dict(zip(sites.site, sites.utc_offset_min))
print(f"  offsets learned for {len(OFFSETS)}/36 sites; "
      f"agree with sites.csv: {all(OFFSETS[s] == prov[s] for s in OFFSETS)}")
OFFSETS = prov            # provided table covers all sites

# F2 — compact 14-digit timestamp; is it UTC?
sub = lab[lab.fmt == "F2"]
g2 = SIGS["F2"]
LEARN["F2_status"] = learn_status(sub, lambda r: g2.match(r).group(7), "F2")
for r, cd in zip(sub.raw_line, sub.c_dev):
    TYPEMAP.setdefault(g2.match(r).group(1), cd.split("-")[0])


def f2_ts(r):
    t = g2.match(r).group(4)
    return pd.Timestamp(f"{t[:4]}-{t[4:6]}-{t[6:8]} {t[8:10]}:{t[10:12]}:{t[12:]}")


off2 = learn_offset(sub, f2_ts)
print(f"  type-char map: {TYPEMAP}; F2 offsets observed: {set(off2.values())} (0 = UTC)")

# F3 — key=value, date order unknown
sub = lab[lab.fmt == "F3"].head(4000)
hits = {"DD/MM": 0, "MM/DD": 0}
for raw, cts in zip(sub.raw_line, sub.c_ts_dt):
    kv = f3_kv(raw)
    t = kv["time"]
    a, b, y, hm = t[:2], t[3:5], t[6:10], t[11:]
    for order, (dd, mm) in {"DD/MM": (a, b), "MM/DD": (b, a)}.items():
        try:
            loc = pd.Timestamp(f"{y}-{mm}-{dd} {hm}:00")
            if loc - pd.Timedelta(minutes=OFFSETS[kv["site"]]) == cts:
                hits[order] += 1
        except ValueError:
            pass
F3_ORDER = max(hits, key=hits.get)
print(f"  F3 date-order evidence: {hits} -> {F3_ORDER}")
sub = lab[lab.fmt == "F3"]
LEARN["F3_status"] = learn_status(sub, lambda r: f3_kv(r)["flag"], "F3")

# F4 — fixed width, integer-looking numerics
sub = lab[lab.fmt == "F4"]
g4 = SIGS["F4"]
LEARN["F4_status"] = learn_status(sub, lambda r: g4.match(r).group(8), "F4")
LEARN["F4_tc"] = learn_num(sub, lambda r: g4.match(r).group(6), sub.c_tc, "F4", "temp")
LEARN["F4_pk"] = learn_num(sub, lambda r: g4.match(r).group(7), sub.c_pk, "F4", "pres")
for r, cd in zip(sub.raw_line, sub.c_dev):
    TYPEMAP.setdefault(g4.match(r).group(1), cd.split("-")[0])

# F5 — US CSV, 12-hour clock, date order unknown
sub = lab[lab.fmt == "F5"].head(4000)
g5 = SIGS["F5"]
hits5 = {"DD/MM": 0, "MM/DD": 0}
for raw, cts in zip(sub.raw_line, sub.c_ts_dt):
    m = g5.match(raw)
    t = m.group(4)
    a, b, y, hh, mi, ap = t[:2], t[3:5], t[6:8], int(t[9:11]), t[12:14], t[14:16]
    h24 = (0 if hh == 12 else hh) + (12 if ap == "PM" else 0)
    for order, (dd, mm) in {"DD/MM": (a, b), "MM/DD": (b, a)}.items():
        try:
            loc = pd.Timestamp(f"20{y}-{mm}-{dd} {h24:02d}:{mi}:00")
            if loc - pd.Timedelta(minutes=OFFSETS[m.group(1)]) == cts:
                hits5[order] += 1
        except ValueError:
            pass
F5_ORDER = max(hits5, key=hits5.get)
print(f"  F5 date-order evidence: {hits5} -> {F5_ORDER}")
sub = lab[lab.fmt == "F5"]
LEARN["F5_status"] = learn_status(sub, lambda r: g5.match(r).group(7), "F5")
LEARN["F5_tc"] = learn_num(sub, lambda r: g5.match(r).group(5), sub.c_tc, "F5", "temp")
LEARN["F5_pk"] = learn_num(sub, lambda r: g5.match(r).group(6), sub.c_pk, "F5", "pres")

# F6 — JSON with epoch seconds
sub = lab[lab.fmt == "F6"]
g6 = SIGS["F6"]
LEARN["F6_status"] = learn_status(sub, lambda r: g6.match(r).group(7), "F6")
off6 = learn_offset(sub, lambda r: pd.Timestamp(int(g6.match(r).group(4)), unit="s"))
print(f"  F6 epoch offsets observed: {set(off6.values())} (0 = UTC)")
print(f"[{time.time()-T0:.0f}s] all mappings learned")

  F1 status map: {'OK': 'OK', 'WARN': 'WARN', 'FAULT': 'FAULT'}
  F1.temp: F->C (max err 0.0000)


  F1.pres: psi->kPa (max err 0.0000)
  offsets learned for 36/36 sites; agree with sites.csv: True
  F2 status map: {'1': 'WARN', '0': 'OK', '2': 'FAULT'}
  type-char map: {'C': 'CMP', 'F': 'FAN', 'V': 'VLV', 'P': 'PMP'}; F2 offsets observed: {0} (0 = UTC)


  F3 date-order evidence: {'DD/MM': 4000, 'MM/DD': 154} -> DD/MM
  F3 status map: {'WARN': 'WARN', 'OK': 'OK', 'FAULT': 'FAULT'}
  F4 status map: {'O': 'OK', 'F': 'FAULT', 'W': 'WARN'}
  F4.temp: x/10 (max err 0.0000)
  F4.pres: x/10 (max err 0.0000)
  F5 date-order evidence: {'DD/MM': 137, 'MM/DD': 4000} -> MM/DD
  F5 status map: {'WRN': 'WARN', 'RUN': 'OK', 'FLT': 'FAULT'}
  F5.temp: F->C (max err 0.0000)
  F5.pres: psi->kPa (max err 0.0000)
  F6 status map: {'ok': 'OK', 'fault': 'FAULT', 'warn': 'WARN'}
  F6 epoch offsets observed: {0} (0 = UTC)
[1s] all mappings learned


## 3 — Assemble the transducer

Every rule below is one of the learned constants; nothing is guessed.
Timestamps from site-local formats (F1/F3/F4/F5) are shifted to UTC with
the per-site offsets (note: shifts can cross midnight and change the date).
Numbers are re-rounded to one decimal *after* the learned transform. Any
line that matches no grammar — or carries a token outside a learned
vocabulary — is REJECT.

In [4]:
def r1(x):
    v = round(x, 1) + 0.0          # normalize -0.0
    return f"{v:.1f}"


def canon(ts, dev, site, tc, pk, st):
    return (f"ts={ts.strftime('%Y-%m-%dT%H:%M:%S')}Z|dev={dev}|site={site}"
            f"|temp_c={tc}|pres_kpa={pk}|status={st}")


NUM = {n: f for n, f in NUM_T.items()}


def transduce(raw):
    fmt = sniff(raw)
    if fmt is None:
        return "REJECT"
    try:
        if fmt == "F1":
            ts_l, dt, dn, site, tf, pp, st = SIGS["F1"].match(raw).groups()
            ts = pd.Timestamp(ts_l) - pd.Timedelta(minutes=OFFSETS[site])
            return canon(ts, f"{dt}-{dn}", site,
                         r1(NUM[LEARN["F1_tc"]](float(tf))),
                         r1(NUM[LEARN["F1_pk"]](float(pp))),
                         LEARN["F1_status"][st])
        if fmt == "F2":
            tch, dn, site, t14, tc, pk, s = SIGS["F2"].match(raw).groups()
            ts = pd.Timestamp(f"{t14[:4]}-{t14[4:6]}-{t14[6:8]} "
                              f"{t14[8:10]}:{t14[10:12]}:{t14[12:]}")
            return canon(ts, f"{TYPEMAP[tch]}-{dn}", site, r1(float(tc)),
                         r1(float(pk)), LEARN["F2_status"][s])
        if fmt == "F3":
            kv = f3_kv(raw)
            m = re.match(r"^([A-Z]{3})(\d{4})$", kv["dev"])
            t = kv["time"]
            a, b, y, hm = t[:2], t[3:5], t[6:10], t[11:]
            dd, mm = (a, b) if F3_ORDER == "DD/MM" else (b, a)
            loc = pd.Timestamp(f"{y}-{mm}-{dd} {hm}:00")
            ts = loc - pd.Timedelta(minutes=OFFSETS[kv["site"]])
            return canon(ts, f"{m.group(1)}-{m.group(2)}", kv["site"],
                         r1(float(kv["temp"][:-1])),
                         r1(float(kv["pressure"][:-3])),
                         LEARN["F3_status"][kv["flag"]])
        if fmt == "F4":
            tch, dn, site, ymd, hm, t10, p10, s = SIGS["F4"].match(raw).groups()
            loc = pd.Timestamp(f"20{ymd[:2]}-{ymd[2:4]}-{ymd[4:6]} "
                               f"{hm[:2]}:{hm[2:]}:00")
            ts = loc - pd.Timedelta(minutes=OFFSETS[site])
            return canon(ts, f"{TYPEMAP[tch]}-{dn}", site,
                         r1(NUM[LEARN["F4_tc"]](int(t10))),
                         r1(NUM[LEARN["F4_pk"]](int(p10))),
                         LEARN["F4_status"][s])
        if fmt == "F5":
            site, dt, dn, t, tf, pp, s = SIGS["F5"].match(raw).groups()
            a, b, y = t[:2], t[3:5], t[6:8]
            hh, mi, ap = int(t[9:11]), t[12:14], t[14:16]
            dd, mm = (a, b) if F5_ORDER == "DD/MM" else (b, a)
            h24 = (0 if hh == 12 else hh) + (12 if ap == "PM" else 0)
            loc = pd.Timestamp(f"20{y}-{mm}-{dd} {h24:02d}:{mi}:00")
            ts = loc - pd.Timedelta(minutes=OFFSETS[site])
            return canon(ts, f"{dt}-{dn}", site,
                         r1(NUM[LEARN["F5_tc"]](float(tf))),
                         r1(NUM[LEARN["F5_pk"]](float(pp))),
                         LEARN["F5_status"][s])
        if fmt == "F6":
            site, dt, dn, e, tc, pk, s = SIGS["F6"].match(raw).groups()
            return canon(pd.Timestamp(int(e), unit="s"), f"{dt}-{dn}", site,
                         r1(float(tc)), r1(float(pk)), LEARN["F6_status"][s])
    except (ValueError, KeyError, AttributeError):
        return "REJECT"
    return "REJECT" 

## 4 — Validate on held-out training pairs

Score with the task's own metric (mean per-line field accuracy) on a 6,000-
line held-out sample, and compare against a **quirk-blind** parser that
recognizes all six shapes but skips the learned corrections (treats local
times as UTC, assumes DD/MM everywhere, reads the 12-hour clock as 24-hour,
takes F4's integers at face value, and copies unmatched lines instead of
rejecting). The gap is the price of the planted quirks.

In [5]:
FIELDS = ["ts", "dev", "site", "temp_c", "pres_kpa", "status"]


def parse_out(s):
    pieces = str(s).split("|")
    kv = {}
    for p in pieces:
        if "=" in p:
            k, v = p.split("=", 1)
            kv.setdefault(k, v)
    return kv, len(pieces)


def field_accuracy(pred, truth):
    total = 0.0
    for p, t in zip(pred, truth):
        p = str(p).strip()
        if t == "REJECT":
            total += 1.0 if p == "REJECT" else 0.0
            continue
        if not p or p == "REJECT":
            continue
        tkv, _ = parse_out(t)
        pkv, n = parse_out(p)
        total += sum(pkv.get(k) == tkv[k] for k in FIELDS) / max(6, n)
    return total / len(truth)


def transduce_blind(raw):
    fmt = sniff(raw)
    if fmt is None:
        return raw
    try:
        if fmt == "F1":
            ts_l, dt, dn, site, tf, pp, st = SIGS["F1"].match(raw).groups()
            return canon(pd.Timestamp(ts_l), f"{dt}-{dn}", site,
                         r1((float(tf) - 32) * 5 / 9),
                         r1(float(pp) * 6.894757), st)
        if fmt == "F2":
            tch, dn, site, t14, tc, pk, s = SIGS["F2"].match(raw).groups()
            ts = pd.Timestamp(f"{t14[:4]}-{t14[4:6]}-{t14[6:8]} "
                              f"{t14[8:10]}:{t14[10:12]}:{t14[12:]}")
            return canon(ts, f"{TYPEMAP[tch]}-{dn}", site, r1(float(tc)),
                         r1(float(pk)), LEARN["F2_status"][s])
        if fmt == "F3":
            kv = f3_kv(raw)
            t = kv["time"]
            loc = pd.Timestamp(f"{t[6:10]}-{t[3:5]}-{t[:2]} {t[11:]}:00")
            return canon(loc, f"{kv['dev'][:3]}-{kv['dev'][3:]}", kv["site"],
                         r1(float(kv["temp"][:-1])),
                         r1(float(kv["pressure"][:-3])), kv["flag"])
        if fmt == "F4":
            tch, dn, site, ymd, hm, t10, p10, s = SIGS["F4"].match(raw).groups()
            loc = pd.Timestamp(f"20{ymd[:2]}-{ymd[2:4]}-{ymd[4:6]} "
                               f"{hm[:2]}:{hm[2:]}:00")
            return canon(loc, f"{TYPEMAP[tch]}-{dn}", site, r1(float(t10)),
                         r1(float(p10)), LEARN["F4_status"][s])
        if fmt == "F5":
            site, dt, dn, t, tf, pp, s = SIGS["F5"].match(raw).groups()
            loc = pd.Timestamp(f"20{t[6:8]}-{t[3:5]}-{t[:2]} "
                               f"{t[9:11]}:{t[12:14]}:00")
            return canon(loc, f"{dt}-{dn}", site, r1((float(tf) - 32) * 5 / 9),
                         r1(float(pp) * 6.894757), LEARN["F5_status"][s])
        if fmt == "F6":
            site, dt, dn, e, tc, pk, s = SIGS["F6"].match(raw).groups()
            return canon(pd.Timestamp(int(e), unit="s"), f"{dt}-{dn}", site,
                         r1(float(tc)), r1(float(pk)), LEARN["F6_status"][s])
    except (ValueError, KeyError, AttributeError):
        return raw
    return raw


rng = np.random.default_rng(SEED)
hold = train.iloc[np.sort(rng.choice(len(train), 6000, replace=False))]
pred_ref = hold.raw_line.map(transduce)
pred_blind = hold.raw_line.map(transduce_blind)
print(f"held-out exact-match (reference): {(pred_ref == hold.canonical).mean():.4%}")
print(f"held-out field accuracy (reference):   {field_accuracy(pred_ref, hold.canonical):.4f}")
print(f"held-out field accuracy (quirk-blind): {field_accuracy(pred_blind, hold.canonical):.4f}")
print(f"held-out field accuracy (all-REJECT):  {field_accuracy(['REJECT']*len(hold), hold.canonical):.4f}")
ok = hold.assign(ok=(pred_ref == hold.canonical).values)
print(ok.groupby(ok.fmt.fillna('corrupted')).ok.mean().round(5).to_string())

held-out exact-match (reference): 100.0000%
held-out field accuracy (reference):   1.0000
held-out field accuracy (quirk-blind): 0.7460
held-out field accuracy (all-REJECT):  0.0385
fmt
F1           1.0
F2           1.0
F3           1.0
F4           1.0
F5           1.0
F6           1.0
corrupted    1.0


## 5 — Transduce the test set and write the submission

In [6]:
import os

out = test[["line_id"]].copy()
out["output"] = test.raw_line.map(transduce)

assert len(out) == len(test)
assert out.line_id.is_unique
assert out.output.notna().all() and (out.output.str.len() > 0).all()
print("REJECT rate on test:", round((out.output == "REJECT").mean(), 4))

os.makedirs("./working", exist_ok=True)
out.to_csv("./working/submission.csv", index=False)
print(f"[{time.time()-T0:.0f}s] wrote ./working/submission.csv ({len(out):,} rows)")

REJECT rate on test: 0.0501
[1s] wrote ./working/submission.csv (8,000 rows)


## Remarks

- The transducer reaches **100% exact match on held-out training pairs**;
  every learned rule was verified with zero error, so the remaining risk on
  test is limited to tokens never seen in training (handled by strict
  grammars → REJECT).
- The quirk-blind parser — same six grammars, none of the learned
  corrections — loses ≈0.24 field accuracy. Site-local→UTC conversion,
  the MM/DD order of the US CSV format, its 12-hour clock, and the
  fixed-width format's implied decimals are where naive solutions bleed.
- Everything is deterministic (seeded holdout; pure rule application), CPU
  only, and runs in seconds — far inside the time budget.